# Topico 2: Desigualdade de Processamento de Dados e Cadeias de Markov

Cenario: telefone sem fio / cadeia de canais ruidosos. Enunciado: `propostas/1_Simulacoes.pdf`, secao 2.

## Preparacao

In [ ]:
import sys, pathlib

REPO = pathlib.Path.cwd().parents[2]          # .../<topico>/ -> .../<proposta>/ -> raiz
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
from IPython.display import display

from utils.info_theory import (
    entropy, kl_divergence, mutual_information, counts_to_joint,
)
from utils.simulation import rng, sample_pmf, empirical_pmf
from utils.estimation import bootstrap_ci
from utils.plotting import (
    setup_estilo, salvar_fig, plot_pmf, tabela, tabela_comparacao,
    plot_curvas, plot_estimativa_vs_n,
)
import src.model as model                     # helpers especificos deste topico

setup_estilo()
GEN = rng(20260902)                           # seed fixo -> resultados reproduziveis
FIGURAS = pathlib.Path("figuras")

## Passo 1: Fonte X

Definir a variavel fonte X (um bit ou um pequeno vetor de bits) e sua distribuicao.

In [ ]:
# Passo 1
P_X = np.array([0.5, 0.5])  # P(X=0), P(X=1): bit fonte da mensagem original

H_X = entropy(P_X)
print(f"P(X) = {P_X}")
print(f"H(X) = {H_X:.4f} bits  (teto de I(X; Y_j) ao longo da cadeia)")

plot_pmf(P_X, rotulos=["0", "1"], ylabel="P(X)")
salvar_fig("t2_passo1_fonte_x")


## Passo 2: Cadeia de canais BSC

Modelar cada elo como um BSC de parametro p; gerar X -> Y1 -> ... -> Yk com `model.compose_bsc_chain` (guardar TODOS os estagios).

In [ ]:
# Passo 2
K = 5                 # numero de elos da cadeia (Y1, ..., Y5)
P_LIST = [0.10] * K   # cada elo eh um BSC com esse parametro p (mesmo p em todos os elos por ora)

x_demo = np.array([0, 1, 1, 0, 1])
estagios_demo = model.compose_bsc_chain(x_demo, P_LIST, GEN)
for j, estagio in enumerate(estagios_demo):
    nome = "X" if j == 0 else f"Y{j}"
    print(f"{nome} = {estagio}")


## Passo 3: Propagar N mensagens

Simular a cadeia para N mensagens independentes.

In [ ]:
# Passo 3
N = 20000  # tamanho da amostra usada para estimar I(X; Yj) ao longo da cadeia (Passo 4)

X_N = sample_pmf(P_X, N, GEN)
ESTAGIOS = model.compose_bsc_chain(X_N, P_LIST, GEN)  # [X, Y1, ..., YK], cada um com N mensagens
print(f"N = {N} mensagens propagadas por {K} elos")
for j, estagio in enumerate(ESTAGIOS):
    nome = "X" if j == 0 else f"Y{j}"
    print(f"P({nome}=1) empirico = {estagio.mean():.4f}")


## Passo 4: I(X; Yj) para cada j

Estimar I(X; Y1), ..., I(X; Yk) e verificar que a sequencia e nao crescente. O criterio de "nao crescente" sai dos proprios ICs bootstrap, nao de uma tolerancia fixa.

In [ ]:
# Passo 4
def _i_xy(x, y):
    return mutual_information(counts_to_joint(x, y, size_x=2, size_y=2))


I_ESTIMADO, I_IC_BAIXO, I_IC_ALTO = [], [], []
for Y_j in ESTAGIOS[1:]:
    est, baixo, alto = bootstrap_ci((X_N, Y_j), _i_xy, generator=GEN, paired=True)
    I_ESTIMADO.append(est)
    I_IC_BAIXO.append(baixo)
    I_IC_ALTO.append(alto)

TABELA_I = tabela(
    {
        "elo": [f"I(X;Y{j})" for j in range(1, K + 1)],
        "estimado": I_ESTIMADO,
        "IC_baixo": I_IC_BAIXO,
        "IC_alto": I_IC_ALTO,
    },
    indice="elo",
)
display(TABELA_I)

DIFF_I = np.diff(I_ESTIMADO)
print(f"H(X) = {H_X:.4f} bits  (teto de I(X; Yj) ao longo da cadeia)")
print(f"I(X;Yj) - I(X;Yj-1): {np.round(DIFF_I, 4)}")

# "Nao crescente" precisa ser julgado contra a incerteza da propria estimativa,
# nao contra uma tolerancia fixa escolhida a mao: um limiar frouxo aceitaria um
# aumento real como ruido, e um limiar apertado reprovaria ruido como violacao.
# Criterio: um elo so sobe de verdade se o piso do seu IC ficar acima do teto do
# IC do elo anterior -- ai os dois intervalos nem se tocam.
SUBIDAS = [j + 1 for j in range(1, K) if I_IC_BAIXO[j] > I_IC_ALTO[j - 1]]
print(f"elos com aumento significativo (ICs disjuntos): {SUBIDAS or 'nenhum'}")

## Passo 5: Grafico da degradacao

Plotar I(X; Yj) vs posicao j na cadeia.

In [ ]:
# Passo 5
J_LIST = list(range(1, K + 1))

plot_estimativa_vs_n(
    J_LIST, I_ESTIMADO, I_IC_BAIXO, I_IC_ALTO,
    teorico=H_X, label="I(X;Yj) estimado",
    xlabel="posicao j na cadeia (Yj)", ylabel="bits", logx=False,
)
salvar_fig("t2_passo5_degradacao_i")


## Passo 6: Variar o ruido

Repetir com elos mais/menos ruidosos e observar a velocidade de degradacao.

In [ ]:
# Passo 6
P_CENARIOS = [0.02, 0.05, 0.10, 0.20]  # intensidade do ruido em cada elo, um cenario por vez

registros = []
for p in P_CENARIOS:
    estagios_p = model.compose_bsc_chain(X_N, [p] * K, GEN)
    for j, Y_j in enumerate(estagios_p[1:], start=1):
        I_j = mutual_information(counts_to_joint(X_N, Y_j, size_x=2, size_y=2))
        registros.append({"j": j, "p": p, "I(X;Yj)": I_j})

DF_RUIDO = pd.DataFrame(registros)
display(DF_RUIDO.pivot(index="p", columns="j", values="I(X;Yj)"))

plot_curvas(
    DF_RUIDO, x="j", y="I(X;Yj)", hue="p", teorico=H_X,
    xlabel="posicao j na cadeia (Yj)", ylabel="bits",
)
salvar_fig("t2_passo6_ruido_intensidade")

## Passo 7: Cadeia inspirada em IA

X = rotulo verdadeiro; gerar imagem (`model.class_conditional_gaussian`), comprimir (menos coordenadas), decidir. Estimar I(rotulo; cada estagio) com IC e reamostrar a QUEDA entre estagios consecutivos, que e o que a DPI afirma.

In [ ]:
# Passo 7
N_ROTULOS = 20000      # amostra grande o bastante para o gap entre estagios sair do ruido
DIM = 8                # dimensao da "imagem" (vetor gaussiano por classe)
FATOR_COMPRESSAO = 4   # 8 -> 2 coordenadas apos a compressao
SIGMA_IMG = 1.5
N_BINS = 10            # bins para discretizar as projecoes continuas

P_ROTULO = np.array([0.5, 0.5])
# sinal decai por coordenada: nem todas carregam a mesma informacao sobre a classe,
# entao a media por bloco (Passo 7) tem chance real de perder informacao.
SINAL = np.array([2.0, 1.6, 1.2, 0.8, 0.4, 0.2, 0.1, 0.0])
MEDIAS_ROTULO = np.array([-SINAL, SINAL])

ROTULO = sample_pmf(P_ROTULO, N_ROTULOS, GEN)
IMAGEM = model.class_conditional_gaussian(ROTULO, MEDIAS_ROTULO, SIGMA_IMG, GEN)
REPRESENTACAO = model.compress_mean_blocks(IMAGEM, FATOR_COMPRESSAO)
MEDIAS_COMPRIMIDAS = model.compress_mean_blocks(MEDIAS_ROTULO, FATOR_COMPRESSAO)
PREDICAO = model.nearest_mean_predict(REPRESENTACAO, MEDIAS_COMPRIMIDAS)

acuracia = (PREDICAO == ROTULO).mean()


def _projecao(x, means):
    """Projeta `x` na direcao (media da classe 1 - media da classe 0): estatistica
    suficiente para o classificador gaussiano de medias iguais em variancia."""
    x = np.atleast_2d(np.asarray(x, dtype=float))
    diff = means[1] - means[0]
    return x @ diff / (diff @ diff)


def _discretizar(projecao, n_bins=N_BINS):
    bordas = np.quantile(projecao, np.linspace(0, 1, n_bins + 1)[1:-1])
    return np.digitize(projecao, bordas)


def _i_rotulo(size_y):
    def estatistica(rotulo, estagio):
        return mutual_information(counts_to_joint(rotulo, estagio, size_x=2, size_y=size_y))

    return estatistica


IMAGEM_DISC = _discretizar(_projecao(IMAGEM, MEDIAS_ROTULO))
REPRESENTACAO_DISC = _discretizar(_projecao(REPRESENTACAO, MEDIAS_COMPRIMIDAS))

ESTAGIOS_IA = [
    ("imagem completa", IMAGEM_DISC, N_BINS),
    ("representacao comprimida", REPRESENTACAO_DISC, N_BINS),
    ("predicao final", PREDICAO, 2),
]

linhas, I_ESTAGIOS = [], []
for nome, estagio, size_y in ESTAGIOS_IA:
    est, baixo, alto = bootstrap_ci(
        (ROTULO, estagio), _i_rotulo(size_y), n_boot=600, generator=GEN, paired=True
    )
    I_ESTAGIOS.append(est)
    linhas.append({"estagio": nome, "I(rotulo;estagio)": est, "IC_baixo": baixo, "IC_alto": alto})

display(tabela(linhas, indice="estagio"))
print(f"acuracia da predicao (rotulo vs predicao): {acuracia:.4f}")

# A ordem I(imagem) >= I(representacao) >= I(predicao) so esta comprovada se a
# QUEDA for maior que a incerteza. Comparar dois ICs que se sobrepoem nao
# responde isso: o certo e reamostrar a DIFERENCA nos mesmos indices, porque os
# dois estagios vem das mesmas amostras e os erros sao correlacionados.
def _queda(rotulo, antes, depois, size_antes, size_depois):
    return _i_rotulo(size_antes)(rotulo, antes) - _i_rotulo(size_depois)(rotulo, depois)


for (nome_a, est_a, sy_a), (nome_b, est_b, sy_b) in zip(ESTAGIOS_IA, ESTAGIOS_IA[1:]):
    queda, q_lo, q_hi = bootstrap_ci(
        (ROTULO, est_a, est_b),
        lambda r, a, b, sa=sy_a, sb=sy_b: _queda(r, a, b, sa, sb),
        n_boot=600,
        generator=GEN,
        paired=True,
    )
    veredito = "queda significativa" if q_lo > 0 else "nao resolvida pelos dados"
    print(f"{nome_a} -> {nome_b}: queda = {queda:.4f} bits, IC [{q_lo:.4f}, {q_hi:.4f}] -> {veredito}")

print(f"DPI respeitada na ordem pontual: {bool(I_ESTAGIOS[0] >= I_ESTAGIOS[1] >= I_ESTAGIOS[2])}")

## Passo 8: Colapso de modelo

Simular varias geracoes: a cada geracao, estimar uma distribuicao a partir de uma amostra finita da anterior. Acompanhar a distancia ate p_real ao longo das geracoes, variando o tamanho de amostra por geracao e repetindo cada cenario varias vezes.

In [ ]:
# Passo 8
K_CAT = 6
P_REAL = np.array([0.05, 0.10, 0.15, 0.30, 0.25, 0.15])  # pmf "real" com 6 categorias
N_GERACOES = 15
N_POR_GERACAO = [30, 100, 300]  # o enunciado pede variar o tamanho de amostra por geracao
R_SEEDS = 40                    # a trajetoria e aleatoria: uma so nao representa o processo

# Uma categoria que zera numa geracao nunca mais volta -- a amostragem seguinte
# nao tem como sortea-la. Isso E o colapso, nao um defeito da simulacao. Mas
# D(p_real || p_ger) fica +inf no instante em que a primeira categoria morre,
# porque p_ger passa a ter zero onde p_real e positiva, e a trajetoria se perde
# dali em diante. Por isso acompanhamos tambem a distancia de variacao total,
# que continua finita e bem definida depois da extincao, e registramos quantas
# categorias sobraram vivas.
def _variacao_total(p, q):
    return 0.5 * float(np.abs(np.asarray(p, dtype=float) - np.asarray(q, dtype=float)).sum())


registros = []
for n_ger in N_POR_GERACAO:
    for semente in range(R_SEEDS):
        p_atual = P_REAL.copy()
        for k in range(1, N_GERACOES + 1):
            amostra = sample_pmf(p_atual, n_ger, GEN)
            p_atual = empirical_pmf(amostra, size=K_CAT)  # geracao k+1 treina so na amostra da k
            registros.append({
                "geracao": k,
                "N por geracao": n_ger,
                "semente": semente,
                "D(p_real||p_ger)": kl_divergence(P_REAL, p_atual),
                "variacao total": _variacao_total(P_REAL, p_atual),
                "categorias vivas": int((p_atual > 0).sum()),
            })

DF_GERACOES = pd.DataFrame(registros)

plot_curvas(
    DF_GERACOES, x="geracao", y="variacao total", hue="N por geracao",
    teorico=0.0, errorbar=("ci", 95),
    xlabel="geracao", ylabel="distancia de variacao total ate p_real",
)
salvar_fig("t2_passo8_colapso_modelo")

FINAL = DF_GERACOES[DF_GERACOES["geracao"] == N_GERACOES]
RESUMO = FINAL.groupby("N por geracao").agg(
    **{
        "seeds com categoria extinta (%)": ("categorias vivas", lambda c: 100.0 * (c < K_CAT).mean()),
        "categorias vivas (mediana)": ("categorias vivas", "median"),
        "variacao total (mediana)": ("variacao total", "median"),
        "D finita (mediana)": ("D(p_real||p_ger)", lambda d: float(np.median(d[np.isfinite(d)]))
                               if np.isfinite(d).any() else float("nan")),
    }
)
display(RESUMO)

for n_ger in N_POR_GERACAO:
    d = FINAL.loc[FINAL["N por geracao"] == n_ger, "D(p_real||p_ger)"]
    print(f"N={n_ger:4d} por geracao: D(p_real||p_ger) = +inf em {100 * (~np.isfinite(d)).mean():.1f}% "
          f"das {R_SEEDS} trajetorias na geracao {N_GERACOES}")

# Uma trajetoria concreta, para comparar categoria a categoria com a real.
p_exemplo = P_REAL.copy()
for _ in range(N_GERACOES):
    p_exemplo = empirical_pmf(sample_pmf(p_exemplo, 300, GEN), size=K_CAT)
display(tabela_comparacao([f"categoria {i}" for i in range(K_CAT)], p_exemplo, P_REAL))

## Relatorio

Consolidar resultados, tabelas e figuras em `relatorio.md` (figuras salvas em `figuras/`).